# Identity, Isolation, and Compliance

> **The story.** In 1996, Ravi Sandhu and colleagues published a consolidated role-based access-control model that separated users, roles, permissions, and constraints. In 2010, John Kindervag gave the industry a sharper operational phrase: Zero Trust. Both ideas matter here because Riverside's assistant crosses more boundaries than a normal search box, and no boundary may inherit authority merely because the previous one looked plausible.
>
> **Where you are.** Riverside has a frozen synthetic engagement, source contracts, and a production-shaped platform boundary. The missing proof is end-to-end context: `RISK-RIV-003` says stale group or title assignment can permit forbidden access, and `INC-RIV-003` records the consequence as `SEV-1`. This chapter builds the local mechanism and evidence package without claiming that a local result proves a deployed or legal control.
>
> **Notation.** $T$ - trusted tenant; $U$ - actor; $R_t$ - trusted active roles; $R_q$ - requested roles; $R_e = R_t \cap R_q$ - effective roles; $G$ - allowed region; $P$ - declared purpose; $A$ - assigned titles; $D$ - candidate resource; $\operatorname{allow}(U,D)$ - deterministic authorization decision.

## 0 - The Challenge

> **The mission**: Riverside Editorial Copilot - preserve tenant, title, role, region, and purpose isolation while keeping bounded editorial assistance usable.

**What we know so far:**
- The frozen case requires seven request-context fields in `FACT-RIV-017`.
- EU manuscript policy permits `TEN-RIV-EU` in `REG-UKS`; US content uses `TEN-RIV-US` in `REG-EUS`.
- **But a valid sign-in still does not prove tenant, title, purpose, or tool authority.**

**What's blocking us:** A disabled contractor still has a stale nested editor group. A caller can also request a different tenant, a broader role, or a rights-changing tool. If a downstream component trusts those request fields, authentication becomes accidental authorization.

**What this chapter unlocks:** A reviewable local mechanism and nine scenarios spanning gateway, retrieval, tool, audit, and response boundaries. The notebook executed successfully against the synthetic fixtures and matched the expected fail-closed decisions, then its outputs were cleared. This is local fixture validation, not Azure, identity-provider, customer, legal, privacy, security, residency, or compliance evidence.

```mermaid
flowchart LR
    A["Valid sign-in"] --> B["Caller-supplied authority"]
    B --> C["Failure: stale or forged context"]
    C --> D["Trusted normalization"]
    D --> E["Retrieval and tool policy"]
    E --> F["Audit every branch"]
    F --> G["Minimized response"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

### Evidence boundary and outcomes

| Label | Meaning | What it cannot claim |
|---|---|---|
| `[Local-static]` | Source, contract, or expected-result inspection without execution | Runtime behavior |
| `[Local-measured]` | A named local run produced a result | Customer production behavior |
| `[Modeled]` | Expected design behavior from explicit policy | Observed enforcement |
| `[External validation required]` | Cloud, security, privacy, legal, retention, or operational proof | Closure from this notebook |

By the end you should be able to derive effective roles, build mandatory retrieval filters, authorize concrete tools, audit allows and denies without content, minimize the public response, and name every external validation owner.

**Warning:** Running this notebook later creates `[Local-measured]` evidence only when you record environment, source commit, fixture version, date, result, and limitations. A green local ledger is never a compliance attestation.

In [ ]:
# -- Load frozen and local synthetic contracts ----------------------------
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any
import hashlib
import json


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "AUTHORING_GUIDE.md").is_file() and (candidate / "learning" / "fde" / "shared").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook inside ai-portfolio.")


def load_json(path: Path) -> dict[str, Any]:
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)


ROOT = find_repo_root(Path.cwd().resolve())
CHAPTER = ROOT / "learning" / "fde" / "04-identity-isolation-and-compliance"
SHARED = ROOT / "learning" / "fde" / "shared" / "fixtures"
engagement = load_json(SHARED / "riverside-engagement-v1.json")
source_samples = load_json(SHARED / "riverside-source-samples-v1.json")
expected_facts = load_json(SHARED / "expected-facts-v1.json")
local_fixture = load_json(CHAPTER / "fixtures" / "identity-scenarios-v1.json")
fde04_facts = [fact for fact in expected_facts["facts"] if "FDE-04" in fact["notebook_ids"]]

print(f"LOCAL-STATIC: frozen={engagement['fixture_version']}, scenarios={len(local_fixture['scenarios'])}")
print(f"  FDE-04 expected facts: {len(fde04_facts)}")
print("  no network, IdP, model, vector store, tool, or cloud service was contacted")

## 1 - Failure First: Caller Filters Are Not Authority

The tempting shortcut is to accept `tenant_id`, `role_ids`, and retrieval filters from the request. That makes integration easy and authorization meaningless: the caller can rewrite the facts the policy is supposed to check.

```mermaid
flowchart LR
    A["EU editor identity"] --> B["Request says US tenant"]
    B --> C["Naive request-only check"]
    C --> D["US resource returned"]
    A --> E["Trusted identity record"]
    E --> F["Mismatch detected"]
    F --> G["Deny and audit"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** If the actor keeps the same ID but changes request tenant, region, role, and title to match a US resource, does a request-only check deny, allow, or error?

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Trust request tenant, roles, region, and titles because sign-in succeeded | Authentication proves who presented a credential, not which Riverside resource or action is allowed |
| Right | Rebuild effective context from an active identity and tenant policy | Forged request scope becomes a deny decision with a named boundary |

**Quick Health Check:** The unsafe check must allow the forged US request, the output must say the actor identity was never consulted, and no reader may mistake that expected failure for a valid authorization result.

**Reflection:** The failure is not a missing prompt instruction. The policy has no trusted facts to compare. Gateway normalization is the minimal next control because it can only reduce requested authority.

In [ ]:
# -- Expose the request-only authorization defect -------------------------
resources = {item["resource_id"]: item for item in local_fixture["resources"]}
scenarios = {item["scenario_id"]: item for item in local_fixture["scenarios"]}


def unsafe_request_only_authorized(request: dict[str, Any], resource: dict[str, Any]) -> bool:
    return (
        request["tenant_id"] == resource["tenant_id"]
        and request["region_id"] == resource["region_id"]
        and bool(set(request["role_ids"]) & set(resource["allowed_role_ids"]))
    )


tampered = dict(scenarios["ISO-RIV-002"]["request"])
tampered.update({"tenant_id": "TEN-RIV-US", "region_id": "REG-EUS", "title_ids": ["TITLE-HARBOR"]})
unsafe_result = unsafe_request_only_authorized(tampered, resources["RES-RIV-US-MANUSCRIPT-HARBOR"])
print(f"UNSAFE EXPECTATION: request-only check returns {unsafe_result}")
print("  actor identity never participated in the decision")

## 2 - Gateway Normalization: Requested Authority Can Only Shrink

The gateway joins the actor to an active identity record, checks tenant membership and allowed region, validates purpose and title assignment, and intersects requested roles with active roles.

$$R_e = R_t \cap R_q, \qquad R_q \subseteq R_t$$

The overlap forms effective roles. The subset condition makes an attempted addition a visible escalation, not a silently ignored typo.

```mermaid
flowchart TD
    A["Seven required fields"] --> B{"Identity active?"}
    B -->|no| X["Deny: identity_disabled"]
    B -->|yes| C{"Tenant and region allowed?"}
    C -->|no| Y["Deny"]
    C -->|yes| D{"Purpose and titles allowed?"}
    D -->|no| Z["Deny"]
    D -->|yes| E{"Requested roles subset?"}
    E -->|no| W["Deny: role_escalation"]
    E -->|yes| F["Trusted RequestContext"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style X fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style Y fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style Z fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style W fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** A disabled contractor requests one role that still appears in a stale nested group. Does intersection rescue the request, or must active identity status deny before role math begins?

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Silently discard requested roles that are not currently trusted | Escalation attempts disappear from the decision record |
| Right | Require requested roles to be a subset and deny additions explicitly | The boundary stays observable and fail closed |
| Wrong | Check roles before identity status, tenant, region, purpose, and title | One familiar role can bypass a more specific prohibition |
| Right | Normalize all seven fields in a fixed order | Every deny has the earliest applicable boundary and reason |

**Quick Health Check:** Confirm the seven required fields from `FACT-RIV-017`, deny a disabled identity before group evaluation, reject any requested role outside trusted active roles, and preserve a named denial reason.

**Reflection:** Trusted context is now bounded, but a safe request can still retrieve an unsafe candidate if the backend drops a filter. The next section adds pre-filtering plus post-verification.

In [ ]:
# -- Normalize trusted request context ------------------------------------
constraints = engagement["identity_and_data_constraints"]
required_context = tuple(constraints["required_request_context"])
allowed_purposes = frozenset(constraints["purposes"])
tenant_by_id = {item["tenant_id"]: item for item in constraints["tenants"]}
identity_records = {
    record["payload"]["actor_id"]: record["payload"]
    for record in source_samples["records"]
    if record["payload_type"] == "identity_record"
}


class AuthorizationDenied(Exception):
    def __init__(self, reason: str, boundary: str) -> None:
        super().__init__(reason)
        self.reason = reason
        self.boundary = boundary


@dataclass(frozen=True)
class RequestContext:
    tenant_id: str
    actor_id: str
    role_ids: tuple[str, ...]
    region_id: str
    purpose: str
    title_ids: tuple[str, ...]
    trace_id: str


def active_roles(identity: dict[str, Any]) -> frozenset[str]:
    return frozenset(identity.get("role_ids", identity.get("direct_role_ids", [])))


def normalize_context(request: dict[str, Any]) -> RequestContext:
    missing = [field for field in required_context if field not in request or request[field] in (None, "", [])]
    if missing:
        raise AuthorizationDenied("missing_context", "gateway")
    identity = identity_records.get(request["actor_id"])
    if identity is None or not identity.get("enabled", False):
        raise AuthorizationDenied("identity_disabled", "gateway")
    if request["tenant_id"] not in identity["tenant_ids"]:
        raise AuthorizationDenied("tenant_not_entitled", "gateway")
    tenant = tenant_by_id.get(request["tenant_id"])
    if tenant is None or request["region_id"] not in tenant["allowed_region_ids"]:
        raise AuthorizationDenied("region_not_allowed", "gateway")
    if request["purpose"] not in allowed_purposes:
        raise AuthorizationDenied("purpose_not_allowed", "gateway")
    trusted_roles = active_roles(identity)
    requested_roles = frozenset(request["role_ids"])
    if not requested_roles.issubset(trusted_roles):
        raise AuthorizationDenied("role_escalation", "gateway")
    requested_titles = frozenset(request["title_ids"])
    if not requested_titles.issubset(frozenset(identity.get("title_ids", []))):
        raise AuthorizationDenied("title_not_assigned", "gateway")
    return RequestContext(request["tenant_id"], request["actor_id"], tuple(sorted(requested_roles)), request["region_id"], request["purpose"], tuple(sorted(requested_titles)), request["trace_id"])


print(f"LOCAL-STATIC: required fields={required_context}")

## 3 - Retrieval Isolation: Filter Before, Verify After

Trusted context becomes mandatory backend filters, not optional hints. The orchestrator then verifies every returned record again, limiting the blast radius of a backend defect, stale cache entry, or adapter that dropped a filter.

```mermaid
flowchart LR
    A["Trusted context"] --> B["Mandatory filter"]
    B --> C["Index query"]
    C --> D["Candidates"]
    D --> E{"Post-check"}
    E -->|match| F["Authorized evidence"]
    E -->|mismatch| G["Drop, audit, investigate"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** If the index adapter drops the tenant filter but returns a high-scoring US manuscript to an EU editor, which control must catch it: the prompt, the post-check, or the user interface?

**Warning:** Post-filtering alone is too late because an unauthorized chunk may already have crossed a boundary or entered model context. Filter before retrieval and verify after retrieval.

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Put tenant, region, title, role, and purpose into the query text | Relevance ranking is not an authorization decision |
| Right | Send mandatory structured filters and verify each returned resource | Backend and adapter failures become bounded, auditable denies |
| Wrong | Assume a vector store's filter syntax proves isolation | Cache, index, adapter, and deployment behavior remain untested |
| Right | Keep local mechanism evidence separate from target-backend negative tests | A green fixture run cannot become a cloud security claim |

**Quick Health Check:** The filter must contain tenant, region, purpose, title, role, and active state; the post-check must independently reject every mismatch; unauthorized content must never enter model context.

**Reflection:** Retrieval can now fail closed, but an authorized document does not authorize a side effect. The next boundary evaluates the concrete tool and payload independently of model intent.

In [ ]:
# -- Build retrieval filters and verify returned resources ----------------
def retrieval_filter(context: RequestContext) -> dict[str, Any]:
    return {
        "tenant_id": context.tenant_id,
        "region_id": context.region_id,
        "purpose": context.purpose,
        "title_ids": list(context.title_ids),
        "role_ids": list(context.role_ids),
        "deletion_state": "active",
    }


def authorize_resource(context: RequestContext, resource: dict[str, Any]) -> None:
    if resource["tenant_id"] != context.tenant_id:
        raise AuthorizationDenied("tenant_mismatch", "retrieval")
    if resource["region_id"] != context.region_id:
        raise AuthorizationDenied("region_mismatch", "retrieval")
    if resource["deletion_state"] != "active":
        raise AuthorizationDenied("resource_inactive", "retrieval")
    if context.purpose not in resource["allowed_purposes"]:
        raise AuthorizationDenied("purpose_mismatch", "retrieval")
    if resource["title_id"] is not None and resource["title_id"] not in context.title_ids:
        raise AuthorizationDenied("title_not_assigned", "retrieval")
    if not set(context.role_ids) & set(resource["allowed_role_ids"]):
        raise AuthorizationDenied("role_not_authorized", "retrieval")


print("LOCAL-STATIC: filter uses tenant, region, purpose, title, role, and active state")

## 4 - Tool Authorization: Approval Cannot Create Authority

The model may propose a tool. Deterministic policy decides whether this actor, role, tenant, purpose, and exact payload may cross the boundary. Human confirmation can satisfy an approval requirement for a permitted action; it cannot legalize a prohibited rights or payment mutation.

```mermaid
flowchart TD
    A["Proposed tool + arguments"] --> B["Schema and target"]
    B --> C{"Role and purpose allow?"}
    C -->|no| X["Deny and audit"]
    C -->|yes| D{"Prohibited action?"}
    D -->|yes| X
    D -->|no| E{"Exact approval required?"}
    E -->|missing| Y["Require approval"]
    E -->|satisfied| F["Minimum-scope service identity"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style X fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style Y fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** `ISO-RIV-008` has a valid editor identity and human confirmation. Does that permit `rights.change_contract`?

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Treat model intent or human confirmation as an authorization source | A prohibited action becomes possible through persuasive text or a broad click |
| Right | Evaluate the concrete tool, role, purpose, prohibited actions, and exact payload | Approval can satisfy one bounded condition without widening authority |
| Wrong | Give the orchestrator a broad credential and rely on policy in the prompt | A bypass path inherits more power than the request needs |
| Right | Use a minimum-scope service identity after deterministic authorization | The execution boundary cannot exceed the policy decision |

**Quick Health Check:** Unknown tools, disallowed roles, wrong purposes, and any policy with prohibited actions must deny; missing confirmation may request approval only for an otherwise permitted action.

**Reflection:** Tool policy bounds the side effect, but a deny is not reviewable unless the system records why it stopped. The next section separates protected decision evidence from low-cardinality metrics and public response fields.

In [ ]:
# -- Authorize concrete tools outside model reasoning ---------------------
tool_policies = {item["tool_name"]: item for item in local_fixture["tool_policies"]}


def authorize_tool(context: RequestContext, request: dict[str, Any]) -> None:
    policy = tool_policies.get(request.get("tool_name"))
    if policy is None:
        raise AuthorizationDenied("unknown_tool", "tool")
    if not set(context.role_ids) & set(policy["allowed_role_ids"]):
        raise AuthorizationDenied("tool_not_authorized", "tool")
    if context.purpose not in policy["allowed_purposes"]:
        raise AuthorizationDenied("tool_purpose_mismatch", "tool")
    if policy["prohibited_actions"]:
        raise AuthorizationDenied("prohibited_action", "tool")
    if policy["requires_human_confirmation"] and not request["human_confirmed"]:
        raise AuthorizationDenied("human_confirmation_required", "tool")


print("LOCAL-STATIC: human confirmation never expands the allowed-role set")

## 5 - Audit and Response: Preserve Decisions, Minimize Disclosure

Audit and metrics have different jobs. Restricted audit needs enough protected identity context to reconstruct allow and deny decisions. Metric labels stay bounded and exclude actor, tenant, trace, request, document, prompt, and completion identifiers. Trusted context reaches response assembly to constrain evidence, but the public v1 response does not echo roles or internal filters.

```mermaid
flowchart LR
    A["Decision + context"] --> B["Restricted audit"]
    A --> C["Internal response assembly"]
    C --> D["Authorized answer/refusal"]
    D --> E["Public v1 envelope"]
    A --> F["Allowlisted metrics"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** Which destination may retain a protected actor reference and trace ID: the restricted audit event, the metric label set, or the public response body?

**Warning:** Hashing is pseudonymization, not anonymization. Production audit identity, retention, access, deletion exceptions, and legal-hold design require Security and Legal/Privacy approval.

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Put actor, tenant, trace, document, prompt, or completion IDs in metric labels | Cardinality, cost, and disclosure risk grow together |
| Right | Keep reconstructable decision facts in restricted audit and metrics allowlisted | Investigation and monitoring get different minimum data |
| Wrong | Echo effective roles and backend filters in the public response | Internal policy details become a new disclosure surface |
| Right | Carry context internally and return only answer/refusal, citations, and bounded trace metadata | Response minimization survives both allow and deny paths |

**Quick Health Check:** Audit must record allow and deny reasons without content or credentials; metric labels must exclude identity and request IDs; the public envelope must omit roles, filters, token claims, and backend errors.

**Reflection:** Each boundary now has a local mechanism. The remaining question is compositional: do the nine frozen scenarios reach the expected earliest boundary with no false allow? The ledger answers that only when it is actually run and recorded.

In [ ]:
# -- Create content-free audit and minimized public response --------------
def stable_fingerprint(value: str) -> str:
    return hashlib.sha256(f"riverside-synthetic-audit:{value}".encode()).hexdigest()[:16]


def audit_event(context: RequestContext | None, request: dict[str, Any], decision: str, reason: str, boundary: str) -> dict[str, Any]:
    return {
        "event_type": "authorization_decision",
        "actor_ref": stable_fingerprint(request["actor_id"]),
        "tenant_ref": stable_fingerprint(request["tenant_id"]),
        "trace_id": request["trace_id"],
        "region_id": request["region_id"],
        "purpose": request["purpose"] or "missing",
        "effective_role_ids": list(context.role_ids) if context else [],
        "boundary": boundary,
        "decision": decision,
        "reason": reason,
        "policy_version": local_fixture["fixture_version"],
        "contains_customer_content": False,
        "contains_credentials": False,
    }


def assemble_response(context: RequestContext | None, decision: str, reason: str) -> tuple[dict[str, Any], dict[str, Any]]:
    internal = {
        "decision_context": asdict(context) if context else None,
        "authorization": {"decision": decision, "reason": reason},
    }
    public = {
        "object": "chat.completion",
        "message": "Authorized synthetic response." if decision == "allow" else "I cannot complete that request.",
        "refusal": None if decision == "allow" else "authorization_denied",
        "citations": [],
        "trace": {"trace_id": context.trace_id if context else "redacted-denial-trace"},
    }
    return internal, public


print("LOCAL-STATIC: internal assembly carries context; public response minimizes it")

## 6 - End-to-End Isolation Ledger

The full path normalizes context, authorizes the requested resource or tool, records allow or deny, and assembles a minimized response. A denial stops downstream work.

```mermaid
flowchart LR
    A["Scenario"] --> B["Gateway"]
    B -->|resource| C["Retrieval"]
    B -->|tool| D["Tool policy"]
    B -->|deny| E["Audit deny"]
    C --> F["Audit + response"]
    D --> F
    E --> G["Refusal"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

In [ ]:
# -- Compose the deterministic decision path ------------------------------
def evaluate_scenario(scenario: dict[str, Any]) -> dict[str, Any]:
    request = scenario["request"]
    context: RequestContext | None = None
    decision, reason, boundary = "deny", "uninitialized", "gateway"
    try:
        context = normalize_context(request)
        if request["resource_id"] is not None:
            authorize_resource(context, resources[request["resource_id"]])
            boundary = "retrieval"
        elif request["tool_name"] is not None:
            authorize_tool(context, request)
            boundary = "tool"
        else:
            raise AuthorizationDenied("missing_target", "gateway")
        decision, reason = "allow", "authorized"
    except AuthorizationDenied as error:
        decision, reason, boundary = "deny", error.reason, error.boundary
    audit = audit_event(context, request, decision, reason, boundary)
    internal, public = assemble_response(context, decision, reason)
    return {
        "scenario_id": scenario["scenario_id"],
        "decision": decision,
        "reason": reason,
        "boundary": boundary,
        "matches_expected": decision == scenario["expected_decision"] and reason == scenario["expected_reason"],
        "audit": audit,
        "internal_response": internal,
        "public_response": public,
    }


# This cell is intentionally unexecuted in the committed notebook.
results = [evaluate_scenario(item) for item in local_fixture["scenarios"]]
matched = sum(item["matches_expected"] for item in results)
false_allows = [
    item["scenario_id"]
    for item, case in zip(results, local_fixture["scenarios"])
    if case["expected_decision"] == "deny" and item["decision"] == "allow"
]
for item in results:
    print(f"{item['scenario_id']}: {item['decision']}/{item['reason']} at {item['boundary']} - expected match={item['matches_expected']}")
print(f"LOCAL-MEASURED ONLY AFTER EXECUTION: {matched}/{len(results)} expected decisions")
print(f"  false allows: {false_allows}")
print("  does not prove cloud, customer, residency, privacy, or legal controls")

**Your turn:** Change one synthetic authorization dimension at a time in the next cell. The mechanism should deny when you add a role, change tenant, remove purpose, change region, or request an unassigned title. Do not add production identifiers.

**Quick Health Check**

1. Every deny stops downstream work and emits an audit decision.
2. Every allow uses only active trusted roles and assigned titles.
3. Retrieval is constrained before query and verified after query.
4. Tool confirmation never broadens role or action authority.
5. The public response does not echo token claims, roles, or filters.

In [ ]:
# -- Change one synthetic authorization dimension -------------------------
exercise = dict(scenarios["ISO-RIV-001"]["request"])
# CHANGE THIS: try TEN-RIV-US, REG-EUS, empty purpose, or ROLE-RIGHTS-COUNSEL.
exercise["role_ids"] = ["ROLE-EDITOR"]
exercise_case = {
    "scenario_id": "ISO-RIV-YOUR-TURN",
    "request": exercise,
    "expected_decision": "allow",
    "expected_reason": "authorized",
}
exercise_result = evaluate_scenario(exercise_case)
print(json.dumps({key: exercise_result[key] for key in ("decision", "reason", "boundary")}, indent=2))

## 7 - Review Artifacts: One Control Story, Six Views

The identity flow names boundaries; RBAC names authority; the residency map names data movement; the threat model names abuse paths; controls map mitigations to evidence; the isolation report records expected and observed decisions.

```mermaid
flowchart TD
    A["Identity flow"] --> G["Isolation review"]
    B["RBAC matrix"] --> G
    C["Residency map"] --> G
    D["Threat model"] --> G
    E["Controls matrix"] --> G
    F["Isolation report"] --> G
    G --> H["External validation remains open"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

### Local proof versus external validation

| Claim | This notebook can support | Still required outside |
|---|---|---|
| Context survives local functions | Static inspection; later local run | Deployed token-to-backend trace |
| Cross-tenant/stale-user logic denies | Later local negative run | IdP, gateway, index, cache, tool negatives |
| Source expresses managed identity | Source/IaC review | Exact RBAC scopes and data-plane calls |
| Region policy is represented | Fixture/policy check | Storage, processing, backup, diagnostics, support, subprocessors |
| Telemetry excludes content/IDs | Schema/source inspection | Sample metrics/logs/traces under failure and load |
| Controls are compliant | Nothing; not a local conclusion | Authorized legal/privacy/security review for named scope |

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Treat six completed documents as six effective controls | Artifact presence says nothing about deployment, ownership, or behavior |
| Right | Link each control to local evidence, external evidence, owner, status, and revalidation trigger | Reviewers can see exactly which gate remains open |
| Wrong | Paste production logs, tokens, or customer content into the evidence pack | The review artifact becomes a new data leak |
| Right | Store references and redacted decision metadata only | Evidence remains traceable without duplicating sensitive material |

**Quick Health Check:** All six artifacts must exist, use the same fixture and policy IDs, distinguish expected from observed results, name external owners, and keep `NOT RUN` where no execution occurred.

**Reflection:** Artifact completeness is necessary but not sufficient. The closing roadmap therefore reports what was constructed, what remains unproven, and which exact evidence FDE 05 must inherit.

In [ ]:
# ── Inventory committed artifacts without changing them ─────────────────
artifact_names = [
    "identity-flow.md",
    "rbac-matrix.md",
    "data-flow-residency-map.md",
    "threat-model.md",
    "controls-matrix.md",
    "isolation-test-report.md",
    "notebook-output-record.md",
]
artifact_paths = [CHAPTER / "templates" / name for name in artifact_names]
missing_artifacts = [str(path) for path in artifact_paths if not path.exists()]
assert not missing_artifacts, missing_artifacts
for path in artifact_paths:
    print(f"LOCAL-STATIC: {path.name} is present")
print("  presence is not review quality or control effectiveness")

## 8 - Roadmap, Coverage, and Forward Bridge

```mermaid
flowchart LR
    A["Caller authority"] --> B["Trusted context"]
    B --> C["Filter + verify"]
    C --> D["Tool policy"]
    D --> E["Audit"]
    E --> F["Minimized response"]
    F --> G["Local run record"]
    G --> H["External gate"]
    style A fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

### Roadmap checkpoint

| Riverside blocker | Before | Constructed local mechanism | Current evidence status |
|---|---|---|---|
| Required context | Valid sign-in could be treated as sufficient | Seven fields normalized from active trusted identity and tenant policy | Verified against synthetic fixtures; outputs cleared |
| Stale contractor | Nested editor group could outlive employment | Active status denies before role evaluation | Expected deny for `ISO-RIV-003`; IdP freshness external |
| Cross-tenant retrieval | Caller filters could select a US resource | Pre-filter plus independent candidate verification | Local mechanism only; index/cache enforcement external |
| Tool authority | Human confirmation could be mistaken for permission | Concrete role, purpose, prohibited-action, and approval checks | Local mechanism only; service RBAC external |
| Audit and response | Decisions could leak content or identity into metrics | Content-free decision event and minimized public envelope | Schema/source inspection; destination and retention external |
| Compliance claim | Green local tests could be over-promoted | Local, modeled, and external evidence classes remain separate | No legal, privacy, security, residency, or customer approval |

### Three-tier coverage ledger

| Tier | Techniques | Reason |
|---|---|---|
| Built with executable proof code | Caller-forgery failure, trusted context normalization, retrieval pre-filter/post-check, tool authorization, audit, response minimization, nine-scenario ledger, negative exercise | These mechanisms define the local isolation path |
| Explained and illustrated | Managed identity, metrics cardinality, containment, residency data flow, control ownership | Local code can show boundaries but not deployed behavior |
| Named with a reason | Token validation, IdP revocation SLA, cache partitioning, vector-filter enforcement, private networking, backup/support regions, legal basis, retention/deletion approval | Each requires target-system or authorized legal/privacy/security evidence |

If a named technique is absent from exactly one tier, that is the coverage bug this ledger exists to catch.

**Constructed:** a seven-field context, gateway/retrieval/tool/audit/response mechanisms, two bounded allows and seven expected denies, six completed artifacts, and separate local/external checklists.

**Not proven:** The verified synthetic fixture run did not contact Azure, an IdP, a deployed index, cache, tool service, network, telemetry destination, backup, or production deletion path. No privacy, legal, compliance, security, residency, or customer validation occurred.

### Key takeaways

1. Valid identity is an input to authorization, not the decision.
2. Requested authority may shrink trusted authority; it never expands it.
3. Retrieval needs server filters before search and verification after search.
4. Approval satisfies a bounded rule; it cannot create forbidden authority.
5. Audit decisions, minimize responses, and keep identity out of metric labels.
6. Local proof earns the right to begin external validation, not claim compliance.

**Output handoff:** `templates/isolation-test-report.md` and `templates/notebook-output-record.md` remain `NOT RUN` until an authorized execution records environment, source commit, fixture versions, observed decisions, false allows, false denies, audit evidence, and limitations.

**Forward:** FDE 05 consumes the unresolved regional route, quota, latency, support, and evidence-owner gaps to model service levels and commercial exposure. Isolation failures remain hard release vetoes, never weighted score inputs.